In [10]:
import requests
import pandas as pd
import time
from tqdm import tqdm

API_KEY  = "RGAPI-3e9cea7a-0de9-4496-b43a-db5e4f32eed9"
HEADER   = {"X-Riot-Token": API_KEY}
KR_URL   = "https://kr.api.riotgames.com"
ASIA_URL = "https://asia.api.riotgames.com"

print("설정 완료!")

설정 완료!


In [11]:
res = requests.get(
    "https://ddragon.leagueoflegends.com/cdn/14.10.1/data/ko_KR/champion.json")
champ_data = res.json()['data']
champ_tags = {name: info['tags'][0] for name, info in champ_data.items()}
print(f"챔피언 수: {len(champ_tags)}개")

챔피언 수: 167개


In [12]:
def classify_comp(tag_list):
    tanks    = tag_list.count('Tank')
    assassins= tag_list.count('Assassin')
    mages    = tag_list.count('Mage')
    fighters = tag_list.count('Fighter')
    marksmen = tag_list.count('Marksman')
    if tanks >= 3:     return '전체탱커'
    elif assassins>=2: return '암살자조합'
    elif mages>=3:     return '마법사조합'
    elif fighters>=3:  return '브루저조합'
    elif marksmen>=2:  return '하이퍼캐리'
    else:              return '혼합조합'

In [13]:
def get_match_ids(puuid, count=10):
    url = f"{ASIA_URL}/lol/match/v5/matches/by-puuid/{puuid}/ids"
    res = requests.get(url, headers=HEADER,
                       params={"queue": 420, "count": count})
    if res.status_code == 200:
        return res.json()
    return []

def get_match_full(match_id):
    res1 = requests.get(
        f"{ASIA_URL}/lol/match/v5/matches/{match_id}",
        headers=HEADER)
    res2 = requests.get(
        f"{ASIA_URL}/lol/match/v5/matches/{match_id}/timeline",
        headers=HEADER)

    if res1.status_code != 200 or res2.status_code != 200:
        return None

    match    = res1.json()
    timeline = res2.json()
    frames   = timeline['info']['frames']

    if len(frames) <= 15:
        return None

    frame_15 = frames[15]

    blue_dragons=0; blue_heralds=0; blue_towers=0
    blue_kills=0;   blue_firstBlood=0; blue_voidgrubs=0
    red_dragons=0;  red_heralds=0;  red_towers=0
    red_kills=0;    red_voidgrubs=0
    first_blood_done = False

    participant_kills  = {i: 0 for i in range(1, 11)}
    participant_deaths = {i: 0 for i in range(1, 11)}

    for frame in frames[:16]:
        for event in frame.get('events', []):
            ts = event.get('timestamp', 0)
            if ts > 900000:
                continue
            etype = event.get('type')

            if etype == 'ELITE_MONSTER_KILL':
                monster     = event.get('monsterType', '')
                killer_team = event.get('killerTeamId', 0)
                if monster == 'DRAGON':
                    if killer_team == 100: blue_dragons   += 1
                    else:                  red_dragons    += 1
                elif monster == 'RIFTHERALD':
                    if killer_team == 100: blue_heralds   += 1
                    else:                  red_heralds    += 1
                elif monster == 'HORDE':
                    if killer_team == 100: blue_voidgrubs += 1
                    else:                  red_voidgrubs  += 1

            elif etype == 'BUILDING_KILL':
                killer_team = event.get('teamId', 0)
                if killer_team == 100: blue_towers += 1
                else:                  red_towers  += 1

            elif etype == 'CHAMPION_KILL':
                killer_id = event.get('killerId', 0)
                victim_id = event.get('victimId', 0)
                if killer_id > 0:
                    participant_kills[killer_id]  = participant_kills.get(killer_id, 0) + 1
                if victim_id > 0:
                    participant_deaths[victim_id] = participant_deaths.get(victim_id, 0) + 1
                for p in match['info']['participants']:
                    if p['participantId'] == killer_id:
                        if p['teamId'] == 100: blue_kills += 1
                        else:                  red_kills  += 1
                        break
                if not first_blood_done and killer_id > 0:
                    for p in match['info']['participants']:
                        if p['participantId'] == killer_id:
                            blue_firstBlood = 1 if p['teamId'] == 100 else 0
                            break
                    first_blood_done = True

    win = None
    for team in match['info']['teams']:
        if team['teamId'] == 100:
            win = 1 if team['win'] else 0
            break

    participants = {p['participantId']: p for p in match['info']['participants']}

    lane_data = {}
    for pid, p in participants.items():
        position = p['teamPosition']
        team_id  = p['teamId']
        pf       = frame_15['participantFrames'][str(pid)]
        champ    = p['championName']
        lane_data[f"{team_id}_{position}"] = {
            'gold_15':  pf['totalGold'],
            'cs_15':    pf['minionsKilled'] + pf['jungleMinionsKilled'],
            'kills':    participant_kills.get(pid, 0),
            'deaths':   participant_deaths.get(pid, 0),
            'champion': champ,
            'tag':      champ_tags.get(champ, 'Unknown'),
        }

    positions = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
    row = {
        'match_id': match_id, 'win': win,
        'blue_dragons': blue_dragons, 'blue_heralds': blue_heralds,
        'blue_towers': blue_towers,   'blue_kills': blue_kills,
        'blue_firstBlood': blue_firstBlood, 'blue_voidgrubs': blue_voidgrubs,
        'red_dragons': red_dragons,   'red_heralds': red_heralds,
        'red_towers': red_towers,     'red_kills': red_kills,
        'red_voidgrubs': red_voidgrubs,
    }

    blue_tags = []
    red_tags  = []

    for pos in positions:
        b = lane_data.get(f"100_{pos}", {})
        r = lane_data.get(f"200_{pos}", {})
        p = pos.lower()
        row[f'blue_{p}_gold']     = b.get('gold_15', 0)
        row[f'blue_{p}_cs']       = b.get('cs_15', 0)
        row[f'blue_{p}_kills']    = b.get('kills', 0)
        row[f'blue_{p}_deaths']   = b.get('deaths', 0)
        row[f'blue_{p}_champion'] = b.get('champion', '')
        row[f'blue_{p}_tag']      = b.get('tag', '')
        row[f'red_{p}_gold']      = r.get('gold_15', 0)
        row[f'red_{p}_cs']        = r.get('cs_15', 0)
        row[f'red_{p}_kills']     = r.get('kills', 0)
        row[f'red_{p}_deaths']    = r.get('deaths', 0)
        row[f'red_{p}_champion']  = r.get('champion', '')
        row[f'red_{p}_tag']       = r.get('tag', '')
        row[f'{p}_gold_diff']     = b.get('gold_15', 0) - r.get('gold_15', 0)
        if b.get('tag'): blue_tags.append(b['tag'])
        if r.get('tag'): red_tags.append(r['tag'])

    row['blue_comp'] = classify_comp(blue_tags)
    row['red_comp']  = classify_comp(red_tags)

    return row

print("함수 정의 완료!")

함수 정의 완료!


In [14]:
# 유저 1명 puuid 가져오기
res = requests.get(
    f"{KR_URL}/lol/league/v4/entries/RANKED_SOLO_5x5/DIAMOND/I",
    headers=HEADER, params={"page": 1})
players = res.json()
test_puuid = players[0]['puuid']
test_ids   = get_match_ids(test_puuid, count=5)

test_rows = []
for mid in test_ids:
    row = get_match_full(mid)
    if row:
        test_rows.append(row)
    time.sleep(1)

df_test = pd.DataFrame(test_rows)
print(f"수집: {len(df_test)}개")
print(df_test[['win','blue_dragons','blue_heralds','blue_voidgrubs',
               'blue_towers','blue_kills']].to_string())
print(df_test[['blue_top_kills','blue_jungle_kills','blue_middle_kills',
               'blue_bottom_kills','blue_utility_kills']].to_string())

수집: 5개
   win  blue_dragons  blue_heralds  blue_voidgrubs  blue_towers  blue_kills
0    1             0             0               3            2           7
1    1             0             0               3            0          19
2    1             1             0               3            0          23
3    1             0             0               3            0          10
4    0             0             0               3            0           8
   blue_top_kills  blue_jungle_kills  blue_middle_kills  blue_bottom_kills  blue_utility_kills
0               5                  0                  2                  0                   0
1               5                  7                  3                  1                   3
2               8                  6                  3                  5                   1
3               0                  3                  1                  3                   3
4               3                  3                  1       

In [15]:
def collect_match_ids(target=15000):
    all_ids = set()
    tiers = [
        ('DIAMOND', 'I'), ('DIAMOND', 'II'),
        ('MASTER', 'I'), ('GRANDMASTER', 'I'), ('CHALLENGER', 'I')
    ]

    for tier, division in tiers:
        if len(all_ids) >= target:
            break
        page = 1
        while len(all_ids) < target:
            if tier in ['MASTER', 'GRANDMASTER', 'CHALLENGER']:
                # 마스터 이상은 단일 리그 API 사용
                res = requests.get(
                    f"{KR_URL}/lol/league/v4/{tier.lower()}leagues/by-queue/RANKED_SOLO_5x5",
                    headers=HEADER)
            else:
                res = requests.get(
                    f"{KR_URL}/lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{division}?page={page}",
                    headers=HEADER)

            if res.status_code != 200 or not res.json():
                break

            players = res.json()
            if isinstance(players, dict):
                players = players.get('entries', [])

            for player in players:
                puuid = player.get('puuid')
                if not puuid:
                    continue
                ids = get_match_ids(puuid, count=10)
                all_ids.update(ids)
                time.sleep(0.5)
                if len(all_ids) >= target:
                    break

            page += 1
            print(f"{tier} {division} 페이지 {page} | 매치 ID: {len(all_ids)}개")

    return list(all_ids)

In [19]:
# puuid 바로 사용
puuid = players[0]['puuid']
print(f"puuid: {puuid[:20]}...")

# 매치 ID 5개
test_ids = get_match_ids(puuid, count=5)
print(f"매치 ID: {test_ids}")

# 5판 수집
test_rows = []
for mid in test_ids:
    row = get_match_full(mid)
    if row:
        test_rows.append(row)
    time.sleep(1)

df_test = pd.DataFrame(test_rows)
print(f"\n수집: {len(df_test)}개")
print(df_test[['win','blue_dragons','blue_voidgrubs',
               'blue_towers','blue_kills']].to_string())
print(df_test[['blue_top_kills','blue_jungle_kills','blue_middle_kills',
               'blue_bottom_kills','blue_utility_kills']].to_string())

puuid: TsWo8n7zXb3H04KTif5u...
매치 ID: ['KR_8207920071', 'KR_8207849940', 'KR_8204459133', 'KR_8204433389', 'KR_8204404243']

수집: 5개
   win  blue_dragons  blue_voidgrubs  blue_towers  blue_kills
0    1             0               3            2           7
1    1             0               3            0          19
2    1             1               3            0          23
3    1             0               3            0          10
4    0             0               3            0           8
   blue_top_kills  blue_jungle_kills  blue_middle_kills  blue_bottom_kills  blue_utility_kills
0               5                  0                  2                  0                   0
1               5                  7                  3                  1                   3
2               8                  6                  3                  5                   1
3               0                  3                  1                  3                   3
4               3    

In [20]:
puuid = players[0]['puuid']
test_ids = get_match_ids(puuid, count=5)

test_rows = []
for mid in test_ids:
    row = get_match_full(mid)
    if row:
        test_rows.append(row)
    time.sleep(1)

df_test = pd.DataFrame(test_rows)
print(f"수집: {len(df_test)}개")
print(f"컬럼 수: {len(df_test.columns)}개")
print("\n=== 전체 데이터 ===")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
print(df_test.to_string())

수집: 5개
컬럼 수: 80개

=== 전체 데이터 ===
        match_id  win  blue_dragons  blue_heralds  blue_towers  blue_kills  blue_firstBlood  blue_voidgrubs  red_dragons  red_heralds  red_towers  red_kills  red_voidgrubs  blue_top_gold  blue_top_cs  blue_top_kills  blue_top_deaths blue_top_champion blue_top_tag  red_top_gold  red_top_cs  red_top_kills  red_top_deaths red_top_champion red_top_tag  top_gold_diff  blue_jungle_gold  blue_jungle_cs  blue_jungle_kills  blue_jungle_deaths blue_jungle_champion blue_jungle_tag  red_jungle_gold  red_jungle_cs  red_jungle_kills  red_jungle_deaths red_jungle_champion red_jungle_tag  jungle_gold_diff  blue_middle_gold  blue_middle_cs  blue_middle_kills  blue_middle_deaths blue_middle_champion blue_middle_tag  red_middle_gold  red_middle_cs  red_middle_kills  red_middle_deaths red_middle_champion red_middle_tag  middle_gold_diff  blue_bottom_gold  blue_bottom_cs  blue_bottom_kills  blue_bottom_deaths blue_bottom_champion blue_bottom_tag  red_bottom_gold  red_bottom

In [21]:
df_test.to_csv('lol_test_5games.csv', index=False, encoding='utf-8-sig')
print("저장 완료!")

저장 완료!


In [23]:
def collect_match_ids(target=15000):
    all_ids = set()
    per_tier = target // 5  # 3,000개씩

    tiers = [
        ('DIAMOND', 'I'),
        ('DIAMOND', 'II'),
        ('master', None),
        ('grandmaster', None),
        ('challenger', None)
    ]

    for tier, division in tiers:
        tier_ids = set()

        if division:  # 다이아몬드
            page = 1
            while len(tier_ids) < per_tier:
                res = requests.get(
                    f"{KR_URL}/lol/league/v4/entries/RANKED_SOLO_5x5/{tier.upper()}/{division}?page={page}",
                    headers=HEADER)
                if res.status_code != 200 or not res.json():
                    break
                for player in res.json():
                    puuid = player.get('puuid')
                    if not puuid: continue
                    ids = get_match_ids(puuid, count=10)
                    tier_ids.update(ids)
                    time.sleep(0.5)
                    if len(tier_ids) >= per_tier: break
                page += 1

        else:  # 마스터 이상 (한 번만 요청)
            res = requests.get(
                f"{KR_URL}/lol/league/v4/{tier}leagues/by-queue/RANKED_SOLO_5x5",
                headers=HEADER)
            if res.status_code != 200: continue
            for player in res.json().get('entries', []):
                puuid = player.get('puuid')
                if not puuid: continue
                ids = get_match_ids(puuid, count=10)
                tier_ids.update(ids)
                time.sleep(0.5)
                if len(tier_ids) >= per_tier: break

        all_ids.update(tier_ids)
        print(f"{tier.upper()} {division or ''} 완료 | 누적: {len(all_ids)}개")

    return list(all_ids)

# 본 수집 실행
print("📦 매치 ID 수집 중...")
all_match_ids = collect_match_ids(target=15000)
print(f"✅ 총 매치 ID: {len(all_match_ids)}개\n")

all_data = []
failed   = 0

for match_id in tqdm(all_match_ids[:12000], desc="매치 수집 중"):
    row = get_match_full(match_id)
    if row:
        all_data.append(row)
    else:
        failed += 1
    time.sleep(1.2)

    if len(all_data) % 500 == 0 and len(all_data) > 0:
        pd.DataFrame(all_data).to_csv(
            f'lol_clean_{len(all_data)}.csv',
            index=False, encoding='utf-8-sig')
        print(f"💾 중간 저장: {len(all_data)}게임")

df_final = pd.DataFrame(all_data)
df_final.to_csv('lol_clean_final.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 수집 완료!")
print(f"총 게임: {len(df_final)}개")
print(f"실패:    {failed}개")

📦 매치 ID 수집 중...
DIAMOND I 완료 | 누적: 3007개
DIAMOND II 완료 | 누적: 6004개
MASTER  완료 | 누적: 9008개
GRANDMASTER  완료 | 누적: 11012개
CHALLENGER  완료 | 누적: 11599개
✅ 총 매치 ID: 11599개



매치 수집 중:   6%|▌         | 721/11599 [21:15<5:48:25,  1.92s/it]

💾 중간 저장: 500게임


매치 수집 중:  12%|█▏        | 1379/11599 [41:31<5:27:41,  1.92s/it]

💾 중간 저장: 1000게임


...

💾 중간 저장: 7500게임


매치 수집 중:  95%|█████████▌| 11022/11599 [5:29:51<19:17,  2.01s/it]

💾 중간 저장: 8000게임


매치 수집 중: 100%|██████████| 11599/11599 [5:47:01<00:00,  1.80s/it]



✅ 수집 완료!
총 게임: 8410개
실패:    3189개
